# Experiment 1 — Jev + LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nipundavid/ai-experiments/blob/main/jev/src/experiment_1.ipynb)

This notebook reproduces the `experiment_1.py` workflow: a single Jev call evaluates both a binary decision and a category selection, then stores the results in a LangGraph state.

> Set `TYPESAFE_API_KEY` before running this notebook.

## Overview

This experiment answers two questions about a user query:

1. Is the request simple?
2. Which category does it belong to?

The output is stored in a typed LangGraph state for downstream routing or approval logic.

In [ ]:
import os

# Optional: load a local .env file if you are running this notebook locally.
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


if 'TYPESAFE_API_KEY' not in os.environ:
    raise RuntimeError('Set the TYPESAFE_API_KEY environment variable before running this notebook.')

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_typesafe import Choice, Noul, TypeSafeClassifier

classifier = TypeSafeClassifier(api_key=os.environ['TYPESAFE_API_KEY'])

class State(TypedDict):
    query: str
    is_simple: bool
    category: str

def analyze_query(state: State):
    query = state['query']

    response = classifier.invoke({
        'state': query,
        'questions': {
            'is_simple': Noul(
                instructions=(
                    'Is this a simple question that can be answered directly without retrieval, '
                    'multiple reasoning steps, or external tools?'
                )
            ),
            'category': Choice(
                instructions=(
                    'Classify the users query into exactly one of the available categories.'
                ),
                criteria={
                    'rag': (
                        'The query is primarily about RAG, retrieval, embeddings, vector databases, '
                        'or RAG architecture.'
                    ),
                    'coding': (
                        'The query primarily asks about writing, debugging, or understanding code.'
                    ),
                    'system_design': (
                        'The query primarily asks about designing software or system architecture.'
                    ),
                    'general': (
                        'The query does not clearly belong to any of the other categories.'
                    ),
                },
            ),
        },
    })

    print('\n' + '=' * 70)
    print('RAW JEV RESPONSE')
    print('=' * 70)
    print(response)

    return {
        'is_simple': response.nouls['is_simple'].noul,
        'category': response.choices['category'].choice,
    }

graph = StateGraph(State)
graph.add_node('analyze_query', analyze_query)
graph.add_edge(START, 'analyze_query')
graph.add_edge('analyze_query', END)
app = graph.compile()

In [ ]:
query = 'Explain how RAG works with OpenSearch and embeddings'
result = app.invoke({'query': query})

print('\n' + '=' * 70)
print('FINAL LANGGRAPH STATE')
print('=' * 70)
print(f'\nQuery:\n{result['query']}')
print(f'\nIs simple:\n{result['is_simple']}')
print(f'\nCategory:\n{result['category']}')

## Result interpretation

This notebook is intentionally simple: it validates that a single Jev request can return multiple structured decisions in one response and that those values can be used directly inside a LangGraph workflow.